# PlantVillage: Feature Caching Pipeline
**Purpose:** Extract and cache EfficientNetV2B0 features from raw images.
**Output:** Saved features in Google Drive for model training

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
ZIP_PATH = "/content/drive/MyDrive/PlantVillage_Project/data/raw/color.zip"
!mkdir -p /content/data/raw

!unzip -q "{ZIP_PATH}" -d /content/data/raw

print("Unzipped folder contents")
!ls /content/data/raw/color

Unzipped folder contents
 Apple___Apple_scab
 Apple___Black_rot
 Apple___Cedar_apple_rust
 Apple___healthy
 Blueberry___healthy
'Cherry_(including_sour)___healthy'
'Cherry_(including_sour)___Powdery_mildew'
'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot'
'Corn_(maize)___Common_rust_'
'Corn_(maize)___healthy'
'Corn_(maize)___Northern_Leaf_Blight'
 Grape___Black_rot
'Grape___Esca_(Black_Measles)'
 Grape___healthy
'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)'
'Orange___Haunglongbing_(Citrus_greening)'
 Peach___Bacterial_spot
 Peach___healthy
 Pepper,_bell___Bacterial_spot
 Pepper,_bell___healthy
 Potato___Early_blight
 Potato___healthy
 Potato___Late_blight
 Raspberry___healthy
 Soybean___healthy
 Squash___Powdery_mildew
 Strawberry___healthy
 Strawberry___Leaf_scorch
 Tomato___Bacterial_spot
 Tomato___Early_blight
 Tomato___healthy
 Tomato___Late_blight
 Tomato___Leaf_Mold
 Tomato___Septoria_leaf_spot
'Tomato___Spider_mites Two-spotted_spider_mite'
 Tomato___Target_Spot
 Tomato___Tom

In [5]:
# 1. SETUP ENVIRONMENT

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import sys
import os
import time
from tqdm import tqdm

# Add project to Python path
sys.path.append('/content/drive/MyDrive/PlantVillage_Project/code/scripts')

# 2. IMPORT AND VERIFY

try:
    from extract_features import extract_and_cache_features
    print("Successfully imported extract_and_cache_features")
except Exception as e:
    print(f" Import failed: {e}")
    raise


# 3. DEFINE PATHS

# Raw data (already unzipped and verified)
data_dir = "/content/data/raw/color"

# Processed features will be saved here
cache_dir = "/content/drive/MyDrive/PlantVillage_Project/data/processed"
os.makedirs(cache_dir, exist_ok=True)

print(f" Data directory: {data_dir}")
print(f" Cache directory: {cache_dir}")


# 4. VERIFY DATA

print(" Verifying data structure...")
!ls -1 "/content/data/raw/color" | head -5  # Show first 5 classes
print(f"Total classes: {len(os.listdir(data_dir))}")

# Count total images
total_images = sum([len(files) for r, d, files in os.walk(data_dir)])
print(f"Total images: {total_images}")

# 5. RUN FEATURE EXTRACTION

print(" Starting feature extraction...")
start_time = time.time()

try:
    features, labels, class_names = extract_and_cache_features(
        data_dir=data_dir,
        cache_dir=cache_dir,
        cache_name="efficientnetv2_features.npz",
        force_reextract=False,  #lets Set to False after first run that dont reapeat the full process again
        img_size=224,
        batch_size=32
    )

    # 6. SHOW RESULTS

    print(f" Feature extraction completed in {time.time() - start_time:.2f} seconds")
    print(f"   Features shape: {features.shape}")
    print(f"   Labels shape: {labels.shape}")
    print(f"   Classes: {len(class_names)}")
    print(f"   Feature dimension: {features.shape[1]}")

    # Verify saved file
    cache_path = os.path.join(cache_dir, "efficientnetv2_features.npz")
    print(f"\n Saved features:")
    print(f"   Path: {cache_path}")
    print(f"   Exists: {os.path.exists(cache_path)}")
    if os.path.exists(cache_path):
        print(f"   Size: {os.path.getsize(cache_path)/1024/1024:.2f} MB")

except Exception as e:
    print(f" Error during feature extraction: {e}")
    print("Please check:")
    print("1. Raw images exist at", data_dir)
    print("2. You have enough storage space in Google Drive")
    print("3. The script doesn't have syntax errors")


Mounted at /content/drive
Successfully imported extract_and_cache_features
 Data directory: /content/data/raw/color
 Cache directory: /content/drive/MyDrive/PlantVillage_Project/data/processed
 Verifying data structure...
Apple___Apple_scab
Apple___Black_rot
Apple___Cedar_apple_rust
Apple___healthy
Blueberry___healthy
Total classes: 38
Total images: 54305
 Starting feature extraction...
Loading cached features from /content/drive/MyDrive/PlantVillage_Project/data/processed/efficientnetv2_features.npz
 Feature extraction completed in 11.71 seconds
   Features shape: (54305, 1280)
   Labels shape: (54305, 38)
   Classes: 38
   Feature dimension: 1280

 Saved features:
   Path: /content/drive/MyDrive/PlantVillage_Project/data/processed/efficientnetv2_features.npz
   Exists: True
   Size: 244.41 MB
